In [1]:
import torch.nn as nn
import torchvision
import torch
from torchinfo import summary
from DWT import DWT_2D
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import random
import pandas as pd
import ast
from scipy import stats
from tqdm import tqdm

import os
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from sklearn.metrics import roc_auc_score, confusion_matrix, accuracy_score, precision_recall_fscore_support
from sklearn.metrics import roc_curve, auc, accuracy_score, confusion_matrix, f1_score, precision_recall_curve, precision_score, recall_score, cohen_kappa_score
from sklearn.utils import resample
import seaborn as sns
from scipy.stats import pearsonr

from IXCM import IXCNet

In [2]:
class TB11KDataset(Dataset):
    def __init__(self, list_path, img_root_path, transform=None):

        self.img_root_path = img_root_path
        self.transform = transform
        self.img_paths = []
        self.labels = []

        with open(list_path, 'r') as f:
            lines = f.readlines()
            for line in lines:
                parts = line.strip().split()
                rel_path = parts[0]
 
                full_path = os.path.join(self.img_root_path, rel_path)
                
                if '/tb/' in rel_path or rel_path.startswith('tb/'):
                    label = 1.0
                else:
                    label = 0.0
                
                self.img_paths.append(full_path)
                self.labels.append(label)

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img_path = self.img_paths[idx]
        label = self.labels[idx]

        try:
            image = Image.open(img_path).convert('L') 
        except Exception as e:
            print(f"Error loading image {img_path}: {e}")

            image = Image.new('L', (256, 256))

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor([label], dtype=torch.float32)

In [3]:
def seed_everything(seed=42):

    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True 
    torch.backends.cudnn.benchmark = False

    print(f"Random Seed set to: {seed}")

In [4]:
def get_preds_and_labels(model, dataloader, device, img_dec=True):
    model.eval()
    all_preds = []
    all_preds_s = []
    all_preds_a = []
    all_labels = []
    
    with torch.no_grad():
        for inputs, labels in tqdm(dataloader, desc="Running Inference"):
            inputs = inputs.to(device)
            labels = labels.to(device).float()

            alphas, betas, outs = model(inputs)
            if img_dec:
                alpha_a = alphas[:, 1::2] 
                beta_a  = betas[:, 1::2]
                
                alpha_s = alphas[:, 0::2]
                beta_s  = betas[:, 0::2]
                
                prob_s = alpha_s / (alpha_s + beta_s + 1e-8)
                prob_a = alpha_a / (alpha_a + beta_a + 1e-8)

                all_preds_s.append(prob_s.cpu().numpy())
                all_preds_a.append(prob_a.cpu().numpy())
            else:
                alpha_s = alphas
                beta_s  = betas
                prob_s = alpha_s / (alpha_s + beta_s + 1e-8)
                all_preds_s.append(prob_s.cpu().numpy())

            all_preds.append(outs.cpu().numpy())
            all_labels.append(labels.cpu().numpy())
            
    y_scores = np.concatenate(all_preds, axis=0)
    y_s_scores = np.concatenate(all_preds_s, axis=0)
    if img_dec:
        y_a_scores = np.concatenate(all_preds_a, axis=0)
    else:
        y_a_scores = np.concatenate([[0]], axis=0)
    y_true = np.concatenate(all_labels, axis=0)
    
    return y_scores, y_s_scores, y_a_scores, y_true

def find_optimal_thresholds(model, val_loader, device, label_columns, mode='youden'):

    assert mode in ['f1', 'youden', 'distance', 'mid'], "Mode must be 'f1' or 'youden' or 'distance'"
    
    print(f"{'='*20} Finding Optimal Thresholds on Val Set (Mode: {mode}) {'='*20}")
    
    y_scores, _, _, y_true = get_preds_and_labels(model, val_loader, device)
    
    optimal_thresholds = []
    best_scores = []
    print("Searching best thresholds for each class...")
    for i, class_name in enumerate(label_columns):
        cls_true = y_true[:, i]
        cls_score = y_scores[:, i]

        if np.sum(cls_true) == 0 or np.sum(cls_true) == len(cls_true):
            print(f"  [Warning] Class {class_name} has no positive/negative samples in Val Set. Using default 0.5")
            optimal_thresholds.append(0.5)
            best_scores.append(0.0)
            continue

        best_threshold = 0.5
        best_score = 0.0

        if mode == 'f1':
            precision, recall, thresholds = precision_recall_curve(cls_true, cls_score)

            numerator = 2 * recall * precision
            denominator = recall + precision + 1e-8
            f1_scores = np.divide(numerator, denominator, out=np.zeros_like(numerator), where=denominator!=0)

            best_idx = np.argmax(f1_scores)

            if best_idx < len(thresholds):
                best_threshold = thresholds[best_idx]
            else:
                best_threshold = 0.5
            
            best_score = f1_scores[best_idx]

        elif mode == 'youden':
            fpr, tpr, thresholds = roc_curve(cls_true, cls_score)

            j_scores = tpr - fpr
            
            best_idx = np.argmax(j_scores)
            
            best_threshold = thresholds[best_idx]
            best_score = j_scores[best_idx]

            if best_threshold > 1.0:
                 best_threshold = 1.0
        elif mode == 'distance':
            fpr, tpr, thresholds = roc_curve(cls_true, cls_score)

            dists = np.sqrt((1 - tpr)**2 + fpr**2)

            best_idx = np.argmin(dists)
            best_threshold = thresholds[best_idx]
            best_score = dists[best_idx]
        elif mode == 'mid':
            best_threshold = 0.5

        optimal_thresholds.append(best_threshold)
        best_scores.append(best_score)
        
        print(f"  Class: {class_name:<20} | Best Threshold: {best_threshold:.4f} | Val {mode}: {best_score:.4f}")
        
    return np.array(optimal_thresholds)
    
def calculate_consistency_bootstrap(scores_main, thresh_main, scores_sub, thresh_sub, 
                                    n_bootstraps=1000, alpha=0.95, seed=42):

    rng = np.random.RandomState(seed)
    n_samples = len(scores_main)

    bin_main = (scores_main > thresh_main).astype(int)
    bin_sub = (scores_sub > thresh_sub).astype(int)
    
    try:
        kappa_orig = cohen_kappa_score(bin_main, bin_sub)
    except:
        kappa_orig = 0.0
        
    boot_kappas = []

    for i in range(n_bootstraps):
        indices = rng.randint(0, n_samples, n_samples)
        
        s_main = scores_main[indices]
        s_sub = scores_sub[indices]

        b_main = (s_main > thresh_main).astype(int)
        b_sub = (s_sub > thresh_sub).astype(int)
        
        try:
            k = cohen_kappa_score(b_main, b_sub)
            if not np.isnan(k):
                boot_kappas.append(k)
        except:
            pass

    boot_kappas = np.array(boot_kappas)
    if len(boot_kappas) == 0:
        return kappa_orig, 0.0, 0.0
        
    boot_kappas.sort()
    
    lower_idx = int((1.0 - alpha) / 2.0 * len(boot_kappas))
    upper_idx = int((1.0 - (1.0 - alpha) / 2.0) * len(boot_kappas))

    lower_idx = max(0, lower_idx)
    upper_idx = min(len(boot_kappas) - 1, upper_idx)
    
    return kappa_orig, boot_kappas[lower_idx], boot_kappas[upper_idx]

def calculate_bootstrap_batch(y_true, y_score, threshold, binary_metrics, prob_metrics, 
                              n_bootstraps=1000, alpha=0.95, seed=42):

    rng = np.random.RandomState(seed)

    y_pred_bin_orig = (y_score > threshold).astype(int)
    results = {}

    for name, func in binary_metrics.items():
        try:
            results[name] = {'original': func(y_true, y_pred_bin_orig), 'bootstraps': []}
        except:
            results[name] = {'original': 0.0, 'bootstraps': []}

    for name, func in prob_metrics.items():
        try:
            results[name] = {'original': func(y_true, y_score), 'bootstraps': []}
        except:
            results[name] = {'original': 0.0, 'bootstraps': []}

    n_samples = len(y_true)
    for i in range(n_bootstraps):
        indices = rng.randint(0, n_samples, n_samples)
        
        y_true_boot = y_true[indices]
        y_score_boot = y_score[indices]

        if len(np.unique(y_true_boot)) < 2:
            continue
            
        y_pred_bin_boot = (y_score_boot > threshold).astype(int)

        for name, func in binary_metrics.items():
            try:
                score = func(y_true_boot, y_pred_bin_boot)
                results[name]['bootstraps'].append(score)
            except ValueError:
                pass

        for name, func in prob_metrics.items():
            try:
                score = func(y_true_boot, y_score_boot)
                results[name]['bootstraps'].append(score)
            except ValueError:
                pass

    final_output = {}
    
    for name, data in results.items():
        original = data['original']
        scores = np.array(data['bootstraps'])
        
        if len(scores) == 0:
            final_output[name] = (original, 0.0, 0.0)
            continue
            
        scores.sort()

        lower_idx = int((1.0 - alpha) / 2.0 * len(scores))
        upper_idx = int((1.0 - (1.0 - alpha) / 2.0) * len(scores))

        lower_idx = max(0, lower_idx)
        upper_idx = min(len(scores) - 1, upper_idx)
        
        ci_lower = scores[lower_idx]
        ci_upper = scores[upper_idx]
        
        final_output[name] = (original, ci_lower, ci_upper)
        
    return final_output

def calc_specificity(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return tn / (tn + fp + 1e-8)

def calc_npv(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return tn / (tn + fn + 1e-8)

def calc_sensitivity(y_true, y_pred):
    return recall_score(y_true, y_pred, zero_division=0)

def calc_precision(y_true, y_pred):
    return precision_score(y_true, y_pred, zero_division=0)

def calc_f1(y_true, y_pred):
    return f1_score(y_true, y_pred, zero_division=0)

def calc_acc(y_true, y_pred):
    return accuracy_score(y_true, y_pred)

def calc_auc(y_true, y_score):
    return roc_auc_score(y_true, y_score)

def evaluate_model_comprehensive(model, dataloader, device, label_columns, save_dir, thresholds=None, img_dec=True, evidence=True):
    model.eval()
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
    
    print(f"\n{'='*20} Final Evaluation & Consistency Analysis {'='*20}")

    y_scores, y_s_scores, y_a_scores, y_true = get_preds_and_labels(model, dataloader, device, img_dec=img_dec)
    
    metrics_data = []
    consistency_data = [] 
    roc_curve_data = {} 

    if thresholds is None:
        print("Using default 0.5 threshold for Main Branch.")
        main_thresholds = np.full(len(label_columns), 0.5)
    else:
        print("Using provided optimal thresholds for Main Branch.")
        main_thresholds = thresholds
        
    fixed_thresholds = np.full(len(label_columns), 0.5)

    if evidence:
        if img_dec:
            eval_configs = [
                ('Main', y_scores, main_thresholds),
                ('Structure', y_s_scores, fixed_thresholds),
                ('Appearance', y_a_scores, fixed_thresholds)
            ]
        else:
            eval_configs = [
                ('Main', y_scores, main_thresholds),
                ('Structure', y_s_scores, fixed_thresholds)
            ]
    else:
        eval_configs = [
            ('Main', y_scores, main_thresholds)
        ]

    plt.figure(figsize=(12, 10))
    colors = sns.color_palette("husl", len(label_columns))
    
    binary_funcs = {
        'Accuracy': calc_acc, 'Sensitivity': calc_sensitivity, 'Specificity': calc_specificity,
        'Precision': calc_precision, 'NPV': calc_npv, 'F1_Score': calc_f1
    }
    prob_funcs = {'AUC': calc_auc}

    for branch_name, scores, current_thresholds_list in eval_configs:
        print(f"\n>>> Evaluating Branch: {branch_name} ...")
        for i, class_name in enumerate(label_columns):
            cls_true = y_true[:, i]
            cls_score = scores[:, i]
            current_threshold = current_thresholds_list[i]
            
            row_data = {'Branch': branch_name, 'Class': class_name, 'Threshold': current_threshold}

            batch_results = calculate_bootstrap_batch(
                cls_true, cls_score, current_threshold, binary_funcs, prob_funcs, n_bootstraps=1000
            )
            for metric_name, (score, lower, upper) in batch_results.items():
                row_data[metric_name] = score
                row_data[f'{metric_name}_CI_Lower'] = lower
                row_data[f'{metric_name}_CI_Upper'] = upper

            cls_pred_bin = (cls_score > current_threshold).astype(int)
            tn, fp, fn, tp = confusion_matrix(cls_true, cls_pred_bin, labels=[0, 1]).ravel()
            row_data.update({'TP': tp, 'TN': tn, 'FP': fp, 'FN': fn})

            if branch_name == 'Main':
                fpr, tpr, roc_thresholds = roc_curve(cls_true, cls_score)
                roc_curve_data[class_name] = {'fpr': fpr, 'tpr': tpr, 'auc': row_data['AUC']}
                plt.plot(fpr, tpr, color=colors[i], lw=2, label=f'{class_name} (AUC={row_data["AUC"]:.4f})')

            metrics_data.append(row_data)

    if evidence:
        print(f"\n>>> Calculating Consistency Metrics (Kappa & Pearson) with Bootstrap ...")
        
        for i, class_name in enumerate(label_columns):
            prob_main = y_scores[:, i]
            prob_s = y_s_scores[:, i]
            if img_dec:
                prob_a = y_a_scores[:, i]
            
            thresh_main = main_thresholds[i]
            thresh_sub = 0.5

            kappa_s, k_s_low, k_s_high = calculate_consistency_bootstrap(
                prob_main, thresh_main, prob_s, thresh_sub, n_bootstraps=1000
            )
            corr_s, _ = pearsonr(prob_main, prob_s)
            
            consistency_data.append({
                'Class': class_name,
                'Comparison': 'S_vs_Main',
                'Kappa': kappa_s,
                'Kappa_CI_Lower': k_s_low,
                'Kappa_CI_Upper': k_s_high,
                'Pearson_Corr': corr_s
            })
            if img_dec:
                kappa_a, k_a_low, k_a_high = calculate_consistency_bootstrap(
                    prob_main, thresh_main, prob_a, thresh_sub, n_bootstraps=1000
                )
                corr_a, _ = pearsonr(prob_main, prob_a)
                
                consistency_data.append({
                    'Class': class_name,
                    'Comparison': 'A_vs_Main',
                    'Kappa': kappa_a,
                    'Kappa_CI_Lower': k_a_low,
                    'Kappa_CI_Upper': k_a_high,
                    'Pearson_Corr': corr_a
                })
            if img_dec:
                print(f"  [{class_name}] Kappa(S,Main)={kappa_s:.3f} | Kappa(A,Main)={kappa_a:.3f}")
            else:
                print(f"  [{class_name}] Kappa(S,Main)={kappa_s:.3f}")

    
    plt.plot([0, 1], [0, 1], 'k--', lw=2, alpha=0.5)
    plt.title('Main Branch ROC Curves'); plt.legend(loc="lower right", fontsize=8)
    plt.savefig(os.path.join(save_dir, 'main_roc.png'), dpi=300); plt.close()

    df_metrics = pd.DataFrame(metrics_data)

    branch_order = pd.CategoricalDtype(['Main', 'Structure', 'Appearance'], ordered=True)
    df_metrics['Branch'] = df_metrics['Branch'].astype(branch_order)
    df_metrics = df_metrics.sort_values(by=['Branch', 'Class'])

    numeric_cols = df_metrics.select_dtypes(include=[np.number]).columns
    macro_avg = df_metrics.groupby('Branch', observed=True)[numeric_cols].mean().reset_index()
    macro_avg['Class'] = 'Macro Average'
    df_final = pd.concat([df_metrics, macro_avg], ignore_index=True)
    
    df_final.to_csv(os.path.join(save_dir, 'final_metrics.csv'), index=False)
    if evidence:
        df_consistency = pd.DataFrame(consistency_data)
        cols = ['Class', 'Comparison', 'Kappa', 'Kappa_CI_Lower', 'Kappa_CI_Upper', 'Pearson_Corr']
        df_consistency = df_consistency[cols]
        df_consistency.to_csv(os.path.join(save_dir, 'consistency_analysis.csv'), index=False)
        
        print(f"Evaluation Complete. Files saved to {save_dir}")
    else:
        df_consistency = pd.DataFrame(consistency_data)
    return df_final, df_consistency

In [5]:

evidence=False
img_dec=True
seed_everything(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if img_dec:
    BATCH_SIZE = 64
else:
    BATCH_SIZE = 32
IMG_ROOT = ''
TRAIN_LIST = ''
VAL_LIST = ''

data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((512, 512)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ToTensor(),
        transforms.Normalize([0.5], [0.5]) 
    ]),
    'val': transforms.Compose([
        transforms.Resize((512, 512)),
        transforms.ToTensor(),
        transforms.Normalize([0.5], [0.5])
    ]),
}

train_dataset = TB11KDataset(TRAIN_LIST, IMG_ROOT, transform=data_transforms['train'])
val_dataset = TB11KDataset(VAL_LIST, IMG_ROOT, transform=data_transforms['val'])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

print(f"Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}")

Random Seed set to: 42
Train samples: 6600, Val samples: 1800


In [ ]:
backbone = 'mobilenetv2'
result_root_path = f''
val_eval_output_dir = os.path.join(result_root_path, 'val_evaluation_results')
train_eval_output_dir = os.path.join(result_root_path, 'train_evaluation_results')

final_model = IXCNet(backbone_name=backbone, evi_vec_dim=64, wavename='haar', ydim=1, in_channels=1, img_dec=img_dec).to(DEVICE)
final_model.to(DEVICE)
best_model_path = os.path.join(result_root_path, 'best_model_auc.pth')
if os.path.exists(best_model_path):
    print(f"Loading weights from: {best_model_path}")
    final_model.load_state_dict(torch.load(best_model_path))
else:
    print("Warning: Best model not found, using current weights.")

LABEL_COLUMNS = ['TB']
optimal_thresholds = find_optimal_thresholds(
    final_model, 
    val_loader,
    DEVICE, 
    LABEL_COLUMNS,
    mode='mid'
)

df_val_results, df_val_consist = evaluate_model_comprehensive(
    final_model, 
    val_loader, 
    DEVICE, 
    LABEL_COLUMNS, 
    val_eval_output_dir,
    thresholds=optimal_thresholds,
    img_dec=img_dec,
    evidence=evidence
)
df_train_results, df_train_consist = evaluate_model_comprehensive(
    final_model, 
    train_loader, 
    DEVICE, 
    LABEL_COLUMNS, 
    train_eval_output_dir,
    thresholds=optimal_thresholds,
    img_dec=img_dec,
    evidence=evidence
)